In [1]:
# pip install pyomo gurobipy

In [1]:
import pyomo.environ as pyo


# --------------------------
# 1) Create the model
# --------------------------
m = pyo.ConcreteModel()


# --------------------------
# 2) Sets (names of things)
# --------------------------
m.BUS  = pyo.Set(initialize=[1, 2, 3])                               # three buses
# Each line is stored once as a pair (i,j) with i<j
m.LINE = pyo.Set(dimen=2, initialize=[(1, 2), (1, 3), (2, 3)])       # three lines
m.GEN  = pyo.Set(initialize=['G1', 'G2'])                            # two generators


# --------------------------
# 3) Input data (numbers)
# --------------------------
# Demands at each bus (MW)
Pd = {1: 0.0, 2: 100.0, 3: 50.0}

# Line reactances X (p.u.) and symmetric flow limits (MW)
X    = {(1, 2): 0.10, (1, 3): 0.20, (2, 3): 0.10}
Fmax = {(1, 2): 100.0, (1, 3): 100.0, (2, 3): 100.0}

# Convert to susceptance B = 1/X (how strongly flow reacts to angle difference)
B = {ij: 1.0 / X[ij] for ij in X}

# Generator locations, min/max outputs (MW), and simple quadratic cost a*Pg^2 + b*Pg
gen_bus = {'G1': 1, 'G2': 3}
Pmin    = {'G1': 0.0,  'G2': 0.0}
Pmax    = {'G1': 200.0,'G2': 150.0}
a       = {'G1': 0.01, 'G2': 0.02}
b       = {'G1': 10.0, 'G2': 20.0}

# Wrap data as Pyomo Params (optional but tidy)
m.Pd      = pyo.Param(m.BUS,  initialize=Pd)
m.B       = pyo.Param(m.LINE, initialize=B)
m.Fmax    = pyo.Param(m.LINE, initialize=Fmax)
m.gen_bus = pyo.Param(m.GEN,  initialize=gen_bus)
m.Pmin    = pyo.Param(m.GEN,  initialize=Pmin)
m.Pmax    = pyo.Param(m.GEN,  initialize=Pmax)
m.a       = pyo.Param(m.GEN,  initialize=a)
m.b       = pyo.Param(m.GEN,  initialize=b)


# --------------------------
# 4) Decision variables
# --------------------------
# Generator outputs (bounded by Pmin/Pmax)
m.Pg = pyo.Var(m.GEN, bounds=lambda m,g: (m.Pmin[g], m.Pmax[g]))
# Comment: bounds can be defined as tuples or via (inline) functions that return tuples

# Bus voltage angles (radians). We'll (later) fix bus 1 to 0 as a reference.
m.theta = pyo.Var(m.BUS)

# Line flows (MW), limited in both directions by Fmax
m.F = pyo.Var(m.LINE, bounds=lambda m, i, j: (-m.Fmax[i,j], m.Fmax[i,j]))

# Fix slack/reference angle so the angles have an absolute reference
m.theta[1].fix(0.0)


# --------------------------
# 5) Constraints (the rules)
# --------------------------
# a) Physics on each line: F_ij = B_ij * (theta_i - theta_j)
def flow_rule(m, i, j):
    return m.F[i, j] == m.B[i, j] * (m.theta[i] - m.theta[j])
m.FlowDef = pyo.Constraint(m.LINE, rule=flow_rule)

# b) Supply-demand balance at every bus (Kirchhoff): gen - demand = net outflow
def balance_rule(m, bus):
    # total generation connected to this bus
    gen_at_bus = sum(m.Pg[g] for g in m.GEN if m.gen_bus[g] == bus)
    # flows leaving and entering this bus, from our one-direction line list
    flow_out   = sum(m.F[bus, j] for (i, j) in m.LINE if i == bus)
    flow_in    = sum(m.F[i, bus] for (i, j) in m.LINE if j == bus)
    return gen_at_bus - m.Pd[bus] == flow_out - flow_in
m.NodalBalance = pyo.Constraint(m.BUS, rule=balance_rule)


# --------------------------
# 6) Objective: minimize total cost
# --------------------------
def total_cost(m):
    return sum(m.a[g] * m.Pg[g]**2 + m.b[g] * m.Pg[g] for g in m.GEN)
m.Obj = pyo.Objective(rule=total_cost, sense=pyo.minimize)


# --------------------------
# 7) Solve and print results
# --------------------------
solver = pyo.SolverFactory("gurobi")  # change to "highs" if you don't have Gurobi
res = solver.solve(m, tee=True)

print("\n=== Optimal Solution ===")
print(f"Total cost: {pyo.value(m.Obj):.3f}")
print("Generator dispatch (MW):")
for g in m.GEN:
    print(f"  {g} @ bus {int(pyo.value(m.gen_bus[g]))}: {pyo.value(m.Pg[g]):.3f}")

print("\nBus angles (rad):")
for i in m.BUS:
    print(f"  theta[{i}] = {pyo.value(m.theta[i]):.6f}")

print("\nLine flows (MW):")
for (i, j) in m.LINE:
    print(f"  F[{i}-{j}] = {pyo.value(m.F[i,j]):.3f} (limit ±{pyo.value(m.Fmax[i,j])})")


Read LP format model from file /tmp/tmp3fo5613s.pyomo.lp
Reading time = 0.00 seconds
x1: 6 rows, 7 columns, 15 nonzeros
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (linux64 - "Ubuntu 24.04.3 LTS")

CPU model: Intel(R) Core(TM) Ultra 7 265, instruction set [SSE2|AVX|AVX2]
Thread count: 20 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 6 rows, 7 columns and 15 nonzeros
Model fingerprint: 0xcb102b90
Model has 2 quadratic objective terms
Coefficient statistics:
  Matrix range     [1e+00, 1e+01]
  Objective range  [1e+01, 2e+01]
  QObjective range [2e-02, 4e-02]
  Bounds range     [1e+02, 2e+02]
  RHS range        [5e+01, 1e+02]
Presolve removed 5 rows and 5 columns
Presolve time: 0.00s
Presolved: 1 rows, 2 columns, 2 nonzeros
Presolved model has 2 quadratic objective terms
Ordering time: 0.00s

Barrier statistics:
 AA' NZ     : 0.000e+00
 Factor NZ  : 1.000e+00
 Factor Ops : 1.000e+00 (less than 1 second per iteration)
 Threads    : 1

           

In [2]:
dir(m.theta[1])

['_PPRINT_INDENT',
 '__abs__',
 '__add__',
 '__array_ufunc__',
 '__auto_slots__',
 '__autoslot_mappers__',
 '__bool__',
 '__call__',
 '__class__',
 '__deepcopy__',
 '__deepcopy_field__',
 '__deepcopy_state__',
 '__delattr__',
 '__dir__',
 '__div__',
 '__doc__',
 '__eq__',
 '__float__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__iadd__',
 '__idiv__',
 '__imul__',
 '__init__',
 '__init_subclass__',
 '__int__',
 '__ipow__',
 '__isub__',
 '__itruediv__',
 '__le__',
 '__lt__',
 '__module__',
 '__mul__',
 '__ne__',
 '__neg__',
 '__new__',
 '__pos__',
 '__pow__',
 '__radd__',
 '__rdiv__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__rmul__',
 '__rpow__',
 '__rsub__',
 '__rtruediv__',
 '__setattr__',
 '__setstate__',
 '__sizeof__',
 '__slots__',
 '__str__',
 '__sub__',
 '__subclasshook__',
 '__truediv__',
 '__weakref__',
 '_component',
 '_compute_polynomial_degree',
 '_create_objects_for_deepcopy',
 '_domain',
 '_fixed',
 '_index',
 '_lb',


In [3]:
m.gen_bus['G1']

1

In [4]:
?solver

Type:           GUROBIFILE
String form:    <pyomo.solvers.plugins.solvers.GUROBI.GUROBIFILE object at 0x78b022bab3b0>
File:           ~/github/rhtlab-massimiliano/.venv/lib/python3.12/site-packages/pyomo/solvers/plugins/solvers/GUROBI.py
Docstring:      Direct LP/MPS file-based interface to the GUROBI LP/MIP solver
Init docstring: Constructor

In [5]:
pyo.value(m.F[1,2])

99.99999999990972